# H-MSA-ECN: Hierarchical Multi-Scale Attention Emotion Capsule Network

Integrates **H-CapsNet** (hierarchical dual-branch capsule routing) into the **MSA-ECN** pipeline (Multi-Scale CNN → Conformer → Hierarchical Capsules).

**Two-level emotion hierarchy:**
- **Coarse** (3 valence groups): *Neutral* / *Positive* / *Negative*
- **Fine** (8 RAVDESS emotions): neutral, calm, happy, sad, angry, fearful, disgust, surprised

```
Input (B, 3, 64, T)
  └─ MultiScaleCNN
       └─ cnn_to_sequence
            └─ ConformerEncoder
                 ├─ [after layer 2] coarse_repr ──► CoarsePrimary ► DynRouting ► coarse_caps (3)
                 └─ [after layer 4]  fine_repr  ──►  FinePrimary  ► DynRouting ►  fine_caps  (8)
```

In [1]:
# Uncomment if packages are missing
# !pip install librosa torch torchaudio scikit-learn

## 1. Imports

In [2]:
import os
import numpy as np
import librosa

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

print("PyTorch:", torch.__version__)
print("Device :", "cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.10.0+cpu
Device : cpu


## 2. Emotion Maps

Two-level label hierarchy matching H-CapsNet's coarse → fine design.

In [3]:
# Fine labels (8 RAVDESS emotions)
EMOTION_MAP = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised"
}

FINE_TO_IDX = {
    "neutral": 0, "calm": 1, "happy": 2, "sad": 3,
    "angry": 4, "fearful": 5, "disgust": 6, "surprised": 7
}

# Coarse labels (3 valence groups) — mirrors H-CapsNet coarse branch
FINE_TO_COARSE = {
    "neutral": 0, "calm": 0,          # Neutral
    "happy": 1, "surprised": 1,       # Positive
    "sad": 2, "angry": 2,             # Negative
    "fearful": 2, "disgust": 2,
}

NUM_FINE   = 8
NUM_COARSE = 3

print("Fine   classes:", NUM_FINE)
print("Coarse classes:", NUM_COARSE)

Fine   classes: 8
Coarse classes: 3


## 3. Feature Extraction

Extracts a 3-channel spectral tensor **(MFCC + Mel + Chroma)** per file.  
Channels are frequency-padded to the same height (64) before stacking.

In [4]:
def pad_freq(x, target_F):
    """Pad frequency axis (axis=0) of (F, T) array to target_F."""
    F, T = x.shape
    if F == target_F:
        return x
    return np.pad(x, ((0, target_F - F), (0, 0)), mode="constant")


def extract_multichannel_spectral_features(wav_path, sr=16000, n_mfcc=40, n_mels=64):
    """
    Returns tensor shape (3, 64, T):
        channel 0 -> MFCC   (40-dim, padded to 64)
        channel 1 -> Mel    (64-dim)
        channel 2 -> Chroma (12-dim, padded to 64)
    """
    y, sr = librosa.load(wav_path, sr=sr, mono=True)
    if len(y) == 0:
        raise ValueError("Empty audio signal")

    mfcc   = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    mel    = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)

    def norm(x):
        return (x - x.mean()) / (x.std() + 1e-6)

    mfcc, mel, chroma = norm(mfcc), norm(mel), norm(chroma)

    T = min(mfcc.shape[1], mel.shape[1], chroma.shape[1])
    mfcc, mel, chroma = mfcc[:, :T], mel[:, :T], chroma[:, :T]

    target_F = max(mfcc.shape[0], mel.shape[0], chroma.shape[0])  # 64
    mfcc   = pad_freq(mfcc,   target_F)
    mel    = pad_freq(mel,    target_F)
    chroma = pad_freq(chroma, target_F)

    return np.stack([mfcc, mel, chroma], axis=0)  # (3, 64, T)


def load_ravdess_dataset(dataset_root):
    """
    Walks RAVDESS directory tree.
    Returns list of (feature_array, fine_emotion_str, coarse_idx).
    """
    data = []
    for subdir, _, files in os.walk(dataset_root):
        for file in sorted(files):
            if not file.endswith(".wav"):
                continue
            parts = file.split("-")
            if len(parts) < 7 or parts[1] != "01":
                continue
            emotion_code = parts[2]
            if emotion_code not in EMOTION_MAP:
                continue
            emotion   = EMOTION_MAP[emotion_code]
            coarse_id = FINE_TO_COARSE[emotion]
            wav_path  = os.path.join(subdir, file)
            try:
                X = extract_multichannel_spectral_features(wav_path)
                data.append((X, emotion, coarse_id))
                print(f"  OK  {file}  ->  {emotion} (coarse {coarse_id}) | {X.shape}")
            except Exception as e:
                print(f"  ERR {file}: {e}")
    return data


DATASET_ROOT = "ravedess-emotional-speech-audio"
dataset      = load_ravdess_dataset(DATASET_ROOT)

print(f"\nTotal samples: {len(dataset)}")
if dataset:
    X, lbl, c = dataset[0]
    print(f"  Sample shape : {X.shape}")
    print(f"  Fine label   : {lbl}")
    print(f"  Coarse label : {c}")

  OK  03-01-01-01-01-01-01.wav  ->  neutral (coarse 0) | (3, 64, 104)
  OK  03-01-01-01-01-02-01.wav  ->  neutral (coarse 0) | (3, 64, 105)
  OK  03-01-01-01-02-01-01.wav  ->  neutral (coarse 0) | (3, 64, 103)
  OK  03-01-01-01-02-02-01.wav  ->  neutral (coarse 0) | (3, 64, 100)
  OK  03-01-02-01-01-01-01.wav  ->  calm (coarse 0) | (3, 64, 111)
  OK  03-01-02-01-01-02-01.wav  ->  calm (coarse 0) | (3, 64, 113)
  OK  03-01-02-01-02-01-01.wav  ->  calm (coarse 0) | (3, 64, 110)
  OK  03-01-02-01-02-02-01.wav  ->  calm (coarse 0) | (3, 64, 109)
  OK  03-01-02-02-01-01-01.wav  ->  calm (coarse 0) | (3, 64, 116)
  OK  03-01-02-02-01-02-01.wav  ->  calm (coarse 0) | (3, 64, 126)
  OK  03-01-02-02-02-01-01.wav  ->  calm (coarse 0) | (3, 64, 132)
  OK  03-01-02-02-02-02-01.wav  ->  calm (coarse 0) | (3, 64, 126)
  OK  03-01-03-01-01-01-01.wav  ->  happy (coarse 1) | (3, 64, 109)
  OK  03-01-03-01-01-02-01.wav  ->  happy (coarse 1) | (3, 64, 109)
  OK  03-01-03-01-02-01-01.wav  ->  happy (coars

## 4. Dataset & DataLoader

Each sample now carries **both** fine and coarse labels to support hierarchical loss.

In [5]:
class RAVDESSDataset(Dataset):
    """Returns (spectrogram, fine_idx, coarse_idx)."""

    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        X, emotion, coarse_id = self.data[idx]
        X          = torch.tensor(X, dtype=torch.float32)
        fine_idx   = torch.tensor(FINE_TO_IDX[emotion],  dtype=torch.long)
        coarse_idx = torch.tensor(coarse_id,              dtype=torch.long)
        return X, fine_idx, coarse_idx


def pad_collate_fn(batch):
    """Pads variable-length spectrograms along the time axis."""
    Xs, fine_ys, coarse_ys = zip(*batch)
    max_T  = max(x.shape[-1] for x in Xs)
    padded = [F.pad(x, (0, max_T - x.shape[-1])) for x in Xs]
    return (
        torch.stack(padded),       # (B, 3, 64, T_max)
        torch.stack(fine_ys),      # (B,)
        torch.stack(coarse_ys)     # (B,)
    )


# Stratified split by fine label
fine_labels = [FINE_TO_IDX[d[1]] for d in dataset]
train_data, test_data = train_test_split(
    dataset, test_size=0.2, random_state=42, stratify=fine_labels
)

train_ds = RAVDESSDataset(train_data)
test_ds  = RAVDESSDataset(test_data)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,
                          collate_fn=pad_collate_fn, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=8, shuffle=False,
                          collate_fn=pad_collate_fn, num_workers=0)

print(f"Train batches : {len(train_loader)}")
print(f"Test  batches : {len(test_loader)}")

Train batches : 288
Test  batches : 72


## 5. Model Building Blocks

### 5-A. Multi-Scale CNN

In [6]:
class MultiScaleCNN(nn.Module):
    """
    Three parallel conv branches (3x3, 5x5, 7x7) capture spectro-temporal
    patterns at different scales, then fuse via 1x1 projection.
    """
    def __init__(self, in_channels=3, base_channels=32, out_channels=128):
        super().__init__()
        self.branch_3x3 = nn.Conv2d(in_channels, base_channels, 3, padding=1)
        self.branch_5x5 = nn.Conv2d(in_channels, base_channels, 5, padding=2)
        self.branch_7x7 = nn.Conv2d(in_channels, base_channels, 7, padding=3)
        self.bn   = nn.BatchNorm2d(base_channels * 3)
        self.proj = nn.Conv2d(base_channels * 3, out_channels, 1)

    def forward(self, x):
        # x: (B, 3, F, T)
        x = torch.cat([self.branch_3x3(x),
                        self.branch_5x5(x),
                        self.branch_7x7(x)], dim=1)
        return self.proj(F.relu(self.bn(x)))


def cnn_to_sequence(x):
    """(B, D, F, T) -> mean-pool over F -> (B, T, D)."""
    return x.mean(dim=2).permute(0, 2, 1)

### 5-B. Conformer Encoder

Returns **two** outputs: `coarse_repr` (shallow) and `fine_repr` (deep), matching H-CapsNet's dual-branch design.

In [7]:
class FeedForward(nn.Module):
    def __init__(self, dim, expansion=4, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim * expansion),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim * expansion, dim),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.net(x)


class ConformerConvModule(nn.Module):
    def __init__(self, dim, kernel_size=31, dropout=0.1):
        super().__init__()
        self.layer_norm      = nn.LayerNorm(dim)
        self.pointwise_conv1 = nn.Conv1d(dim, dim * 2, 1)
        self.glu             = nn.GLU(dim=1)
        self.depthwise_conv  = nn.Conv1d(dim, dim, kernel_size,
                                          padding=kernel_size // 2, groups=dim)
        self.batch_norm      = nn.BatchNorm1d(dim)
        self.activation      = nn.SiLU()
        self.pointwise_conv2 = nn.Conv1d(dim, dim, 1)
        self.dropout         = nn.Dropout(dropout)

    def forward(self, x):
        x = self.layer_norm(x).transpose(1, 2)          # (B, D, T)
        x = self.glu(self.pointwise_conv1(x))
        x = self.dropout(self.pointwise_conv2(
                self.activation(self.batch_norm(self.depthwise_conv(x)))))
        return x.transpose(1, 2)                         # (B, T, D)


class ConformerBlock(nn.Module):
    def __init__(self, dim, num_heads=4, ff_expansion=4,
                 conv_kernel=31, dropout=0.1):
        super().__init__()
        self.ff1       = FeedForward(dim, ff_expansion, dropout)
        self.self_attn = nn.MultiheadAttention(dim, num_heads,
                                               dropout=dropout, batch_first=True)
        self.conv      = ConformerConvModule(dim, conv_kernel, dropout)
        self.ff2       = FeedForward(dim, ff_expansion, dropout)
        self.norm      = nn.LayerNorm(dim)

    def forward(self, x, attn_mask=None):
        x = x + 0.5 * self.ff1(x)
        x = x + self.self_attn(x, x, x, attn_mask=attn_mask)[0]
        x = x + self.conv(x)
        x = x + 0.5 * self.ff2(x)
        return self.norm(x)


class ConformerEncoder(nn.Module):
    """
    Stack of ConformerBlocks.
    Taps an intermediate output (coarse_repr) after `coarse_split` layers
    and returns the full-depth output (fine_repr) — H-CapsNet dual-branch style.
    """
    def __init__(self, dim=128, num_layers=4, num_heads=4,
                 dropout=0.1, coarse_split=2):
        super().__init__()
        self.layers       = nn.ModuleList([
            ConformerBlock(dim, num_heads, dropout=dropout)
            for _ in range(num_layers)
        ])
        self.coarse_split = coarse_split

    def forward(self, x, attn_mask=None):
        coarse_repr = None
        for i, layer in enumerate(self.layers):
            x = layer(x, attn_mask)
            if i == self.coarse_split - 1:
                coarse_repr = x     # shallow -> coarse caps
        return coarse_repr, x       # (B,T,D), (B,T,D)

### 5-C. Capsule Primitives

In [8]:
def squash(v, dim=-1, eps=1e-8):
    """Squashing non-linearity — scales capsule length to (0, 1)."""
    norm_sq = (v ** 2).sum(dim=dim, keepdim=True)
    norm    = torch.sqrt(norm_sq + eps)
    return (norm_sq / (1.0 + norm_sq)) * (v / (norm + eps))


class PrimaryTemporalCapsules(nn.Module):
    """
    Projects Conformer sequence (B, T, D) into capsule vectors
    then aggregates temporally -> (B, N_caps, cap_dim).
    """
    def __init__(self, input_dim, num_capsules=16, capsule_dim=16):
        super().__init__()
        self.num_capsules = num_capsules
        self.capsule_dim  = capsule_dim
        self.proj         = nn.Linear(input_dim, num_capsules * capsule_dim)

    def forward(self, x):
        B, T, _ = x.shape
        u = self.proj(x).view(B, T, self.num_capsules, self.capsule_dim)
        return squash(u.mean(dim=1))   # (B, N_caps, cap_dim)


class DynamicRouting(nn.Module):
    """
    Dynamic routing-by-agreement (Sabour et al. 2017).
    Direct PyTorch port of H-CapsNet's SecondaryCapsule layer.
    """
    def __init__(self, num_input_caps, num_output_caps,
                 input_dim, output_dim, routing_iters=3):
        super().__init__()
        self.routing_iters = routing_iters
        self.W = nn.Parameter(
            0.01 * torch.randn(1, num_input_caps, num_output_caps,
                               output_dim, input_dim))

    def forward(self, u):
        # u: (B, N_in, input_dim)
        B, N_in, _ = u.shape
        N_out       = self.W.size(2)

        u_  = u.unsqueeze(2).unsqueeze(-1)              # (B, N_in, 1, D_in, 1)
        W_  = self.W.expand(B, -1, -1, -1, -1)
        u_hat = torch.matmul(W_, u_).squeeze(-1)        # (B, N_in, N_out, D_out)

        b = torch.zeros(B, N_in, N_out, device=u.device)

        for _ in range(self.routing_iters):
            c     = F.softmax(b, dim=-1)                # (B, N_in, N_out)
            s     = (c.unsqueeze(-1) * u_hat).sum(dim=1)
            v     = squash(s)
            b     = b + (u_hat * v.unsqueeze(1)).sum(dim=-1)

        return v   # (B, N_out, D_out)

### 5-D. Hierarchical Capsule Head  *(H-CapsNet core)*

Two independent capsule branches — one for coarse valence, one for fine emotion —  
each with its own primary capsules and dynamic routing weight matrix.  
This is a direct PyTorch equivalent of H-CapsNet's `SecondaryCapsule_coarse` / `SecondaryCapsule_fine` split.

In [9]:
class HierarchicalCapsuleHead(nn.Module):
    """
    H-CapsNet-style two-level capsule hierarchy.

    Coarse branch  (3 valence groups):
        PrimaryTemporalCapsules -> DynamicRouting -> 3 capsules
    Fine branch    (8 emotions):
        PrimaryTemporalCapsules -> DynamicRouting -> 8 capsules

    Coarse caps receive shallower Conformer features (broad patterns);
    fine caps receive deep features (detailed emotion nuance).
    """
    def __init__(
        self,
        conformer_dim    = 128,
        num_coarse_pcaps = 8,
        coarse_pcap_dim  = 16,
        num_coarse_caps  = 3,
        coarse_cap_dim   = 16,
        num_fine_pcaps   = 16,
        fine_pcap_dim    = 16,
        num_fine_caps    = 8,
        fine_cap_dim     = 16,
        routing_iters    = 3
    ):
        super().__init__()

        # Coarse branch
        self.coarse_primary = PrimaryTemporalCapsules(
            conformer_dim, num_coarse_pcaps, coarse_pcap_dim)
        self.coarse_routing = DynamicRouting(
            num_coarse_pcaps, num_coarse_caps,
            coarse_pcap_dim, coarse_cap_dim, routing_iters)

        # Fine branch
        self.fine_primary = PrimaryTemporalCapsules(
            conformer_dim, num_fine_pcaps, fine_pcap_dim)
        self.fine_routing = DynamicRouting(
            num_fine_pcaps, num_fine_caps,
            fine_pcap_dim, fine_cap_dim, routing_iters)

    def forward(self, coarse_repr, fine_repr):
        """
        coarse_repr : (B, T, D)  intermediate Conformer output
        fine_repr   : (B, T, D)  final Conformer output
        Returns: coarse_caps, fine_caps, coarse_probs, fine_probs
        """
        u_c          = self.coarse_primary(coarse_repr)
        coarse_caps  = self.coarse_routing(u_c)
        coarse_probs = torch.norm(coarse_caps, dim=-1)   # (B, 3)

        u_f        = self.fine_primary(fine_repr)
        fine_caps  = self.fine_routing(u_f)
        fine_probs = torch.norm(fine_caps, dim=-1)       # (B, 8)

        return coarse_caps, fine_caps, coarse_probs, fine_probs

## 6. Full H-MSA-ECN Model

In [10]:
class H_MSA_ECN(nn.Module):
    """
    Hierarchical Multi-Scale Attention Emotion Capsule Network.

    Combines:
      MultiScaleCNN          multi-scale spectral feature extraction
      ConformerEncoder       local + global temporal modelling
      HierarchicalCapsuleHead  H-CapsNet dual-branch capsule hierarchy
    """
    def __init__(
        self,
        num_coarse       = NUM_COARSE,
        num_fine         = NUM_FINE,
        cnn_out_dim      = 128,
        conformer_layers = 4,
        conformer_heads  = 4,
        coarse_split     = 2,
        dropout          = 0.1
    ):
        super().__init__()
        self.cnn = MultiScaleCNN(
            in_channels=3, base_channels=32, out_channels=cnn_out_dim)
        self.conformer = ConformerEncoder(
            dim=cnn_out_dim, num_layers=conformer_layers,
            num_heads=conformer_heads, dropout=dropout,
            coarse_split=coarse_split)
        self.caps_head = HierarchicalCapsuleHead(
            conformer_dim=cnn_out_dim,
            num_coarse_caps=num_coarse,
            num_fine_caps=num_fine)

    def forward(self, x):
        """x: (B, 3, 64, T)  ->  coarse_caps, fine_caps, coarse_probs, fine_probs"""
        x = cnn_to_sequence(self.cnn(x))           # (B, T, D)
        coarse_repr, fine_repr = self.conformer(x)  # (B, T, D) x2
        return self.caps_head(coarse_repr, fine_repr)


# Quick shape check
_model = H_MSA_ECN()
_x     = torch.randn(2, 3, 64, 300)
_cc, _fc, _cp, _fp = _model(_x)
print("coarse_caps  :", _cc.shape)   # (2, 3, 16)
print("fine_caps    :", _fc.shape)   # (2, 8, 16)
print("coarse_probs :", _cp.shape)   # (2, 3)
print("fine_probs   :", _fp.shape)   # (2, 8)
del _model, _x, _cc, _fc, _cp, _fp

coarse_caps  : torch.Size([2, 3, 16])
fine_caps    : torch.Size([2, 8, 16])
coarse_probs : torch.Size([2, 3])
fine_probs   : torch.Size([2, 8])


## 7. Hierarchical Margin Loss

Weighted sum of coarse + fine margin losses — mirrors H-CapsNet's multi-output training.

```
total_loss = fine_loss + coarse_weight * coarse_loss
```

The coarse branch acts as a regulariser; `coarse_weight < 1` keeps the fine branch dominant.

In [11]:
class HierarchicalMarginLoss(nn.Module):
    def __init__(self, m_plus=0.9, m_minus=0.1, lambda_=0.5, coarse_weight=0.3):
        super().__init__()
        self.m_plus        = m_plus
        self.m_minus       = m_minus
        self.lambda_       = lambda_
        self.coarse_weight = coarse_weight

    def _margin_loss(self, probs, labels):
        one_hot  = F.one_hot(labels, probs.size(1)).float()
        loss_pos = one_hot          * F.relu(self.m_plus  - probs).pow(2)
        loss_neg = (1.0 - one_hot) * F.relu(probs - self.m_minus).pow(2)
        return (loss_pos + self.lambda_ * loss_neg).sum(dim=1).mean()

    def forward(self, coarse_probs, fine_probs, coarse_labels, fine_labels):
        fine_loss   = self._margin_loss(fine_probs,   fine_labels)
        coarse_loss = self._margin_loss(coarse_probs, coarse_labels)
        total       = fine_loss + self.coarse_weight * coarse_loss
        return total, fine_loss, coarse_loss

## 8. Training & Evaluation Functions

In [12]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = total_fine = total_coarse = 0.0
    for X, fine_y, coarse_y in loader:
        X, fine_y, coarse_y = X.to(device), fine_y.to(device), coarse_y.to(device)
        optimizer.zero_grad()
        _, _, coarse_probs, fine_probs = model(X)
        loss, fine_l, coarse_l = criterion(coarse_probs, fine_probs, coarse_y, fine_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_loss   += loss.item()
        total_fine   += fine_l.item()
        total_coarse += coarse_l.item()
    n = len(loader)
    return total_loss / n, total_fine / n, total_coarse / n


def eval_epoch(model, loader, device):
    model.eval()
    correct_fine = correct_coarse = total = 0
    with torch.no_grad():
        for X, fine_y, coarse_y in loader:
            X, fine_y, coarse_y = X.to(device), fine_y.to(device), coarse_y.to(device)
            _, _, coarse_probs, fine_probs = model(X)
            correct_fine   += (fine_probs.argmax(1)   == fine_y).sum().item()
            correct_coarse += (coarse_probs.argmax(1) == coarse_y).sum().item()
            total          += fine_y.size(0)
    return correct_fine / total, correct_coarse / total

## 9. Train the Model

In [ ]:
device     = "cuda" if torch.cuda.is_available() else "cpu"
NUM_EPOCHS = 30

model = H_MSA_ECN(
    num_coarse       = NUM_COARSE,
    num_fine         = NUM_FINE,
    cnn_out_dim      = 128,
    conformer_layers = 4,
    coarse_split     = 2,
).to(device)

criterion = HierarchicalMarginLoss(coarse_weight=0.3)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print(f"Parameters : {sum(p.numel() for p in model.parameters()):,}")
print(f"Device     : {device}")
print("-" * 72)
print(f"{'Epoch':>5}  {'Total L':>8}  {'Fine L':>8}  {'Coarse L':>9}  {'Fine Acc':>9}  {'Coarse Acc':>10}")
print("-" * 72)

best_fine_acc = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    total_l, fine_l, coarse_l = train_epoch(
        model, train_loader, optimizer, criterion, device)
    fine_acc, coarse_acc = eval_epoch(model, test_loader, device)
    scheduler.step()

    if fine_acc > best_fine_acc:
        best_fine_acc = fine_acc
        torch.save(model.state_dict(), "h_msa_ecn_best.pth")
        marker = " << best"
    else:
        marker = ""

    print(f"{epoch:5d}  {total_l:8.4f}  {fine_l:8.4f}  {coarse_l:9.4f}  {fine_acc:9.4f}  {coarse_acc:10.4f}{marker}")

print("-" * 72)
print(f"Best fine-emotion accuracy: {best_fine_acc:.4f}")

SyntaxError: f-string: expecting '}' (1513528328.py, line 19)

## 10. Save & Single-File Inference

In [ ]:
torch.save(model.state_dict(), "h_msa_ecn_final.pth")
print("Saved -> h_msa_ecn_final.pth")
print("Saved -> h_msa_ecn_best.pth  (best val acc)")

In [ ]:
IDX_TO_FINE   = {v: k for k, v in FINE_TO_IDX.items()}
IDX_TO_COARSE = {0: "Neutral-group", 1: "Positive", 2: "Negative"}

def predict(wav_path, model, device):
    X = extract_multichannel_spectral_features(wav_path)
    X = torch.tensor(X, dtype=torch.float32).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        _, _, coarse_probs, fine_probs = model(X)
    fine_idx   = fine_probs.argmax(1).item()
    coarse_idx = coarse_probs.argmax(1).item()
    print(f"Fine   emotion : {IDX_TO_FINE[fine_idx]}"
          f"  (conf {fine_probs[0, fine_idx]:.3f})")
    print(f"Coarse valence : {IDX_TO_COARSE[coarse_idx]}"
          f"  (conf {coarse_probs[0, coarse_idx]:.3f})")

# Usage (set a real path):
# predict("path/to/audio.wav", model, device)